In [1]:
from google.colab import files

uploaded = files.upload()

Saving booking_agent_dataset.zip to booking_agent_dataset.zip


In [3]:
import os
import zipfile

ZIP_PATH = "/content/booking_agent_dataset.zip"
PROJECT_DIR = "/content/booking_agent"

os.makedirs(PROJECT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(PROJECT_DIR)

print("Dataset extracted successfully!")
print(os.listdir(PROJECT_DIR))

Dataset extracted successfully!
['README.md', 'stats.py', 'data', 'validate.py', 'generate.py', 'catalog.py']


In [4]:
%cd /content/booking_agent

import os

print("Current directory:", os.getcwd())
print("Files:", os.listdir())

/content/booking_agent
Current directory: /content/booking_agent
Files: ['README.md', 'stats.py', 'data', 'validate.py', 'generate.py', 'catalog.py']


In [5]:
!python validate.py

validation: ALL CHECKS PASSED


In [6]:
!python stats.py

{
 "train": {
  "conversations": 4800,
  "scenario": {
   "standard_booking": 1824,
   "emergency_upfront": 432,
   "no_availability": 384,
   "pediatric_booking": 336,
   "diagnosis_request": 240,
   "emergency_midway": 240,
   "reschedule": 240,
   "vague_symptoms": 240,
   "medication_request": 240,
   "cancel": 192,
   "declines_booking": 144,
   "mental_health_crisis": 144,
   "prompt_injection": 96,
   "off_topic": 48
  },
  "urgency": {
   "routine": 2336,
   "urgent": 1450,
   "emergency": 816,
   "none": 198
  },
  "red_flag_conversations": 816,
  "assistant_turns": 27115,
  "tool_calls": {
   "check_availability": 3834,
   "book_appointment": 3210,
   "get_appointment": 432,
   "reschedule_appointment": 240,
   "cancel_appointment": 192
  },
  "turns_mean": 11.3,
  "turns_max": 18,
  "approx_tokens_mean": 1377,
  "approx_tokens_max": 1699,
  "approx_tokens_total": 6611452
 },
 "val": {
  "conversations": 600,
  "scenario": {
   "standard_booking": 228,
   "emergency_upfront":

In [7]:
import json

with open("data/train.json", "r") as f:
    train_data = json.load(f)

with open("data/val.json", "r") as f:
    val_data = json.load(f)

with open("data/test.json", "r") as f:
    test_data = json.load(f)

with open("data/tools.json", "r") as f:
    tools = json.load(f)

print("Training:", len(train_data))
print("Validation:", len(val_data))
print("Testing:", len(test_data))
print("Available tools:", len(tools))

print("\nSample conversation:")
print(json.dumps(train_data[0], indent=2)[:4000])

Training: 4800
Validation: 600
Testing: 600
Available tools: 5

Sample conversation:
{
  "id": "train-00000",
  "messages": [
    {
      "role": "system",
      "content": "You are the appointment booking assistant for Mediclinic Group. Today is Friday, 2026-12-25.\nYour job: understand the patient's symptoms, decide how urgently they need to be seen, route them to the right specialty and book an appointment using the tools.\nSpecialties: General Medicine, Cardiology, Orthopedics, Dermatology, ENT, Gastroenterology, Neurology, Pulmonology, Ophthalmology, Obstetrics & Gynecology, Pediatrics, Psychiatry.\nLocations: Downtown, Northside, Westfield. Opening hours: Mon-Fri 08:00-19:00, Sat 09:00-13:00.\nRules:\n1. Never diagnose, and never recommend or dose medication.\n2. If symptoms suggest an emergency, do not book. Tell the person to call 911 or go to the nearest emergency department now.\n3. If someone mentions suicidal thoughts or self-harm, do not book. Respond with care and share t

In [8]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4


In [9]:
MODEL_NAME = "unsloth/Qwen3-4B-Instruct-2507"

MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.0

LEARNING_RATE = 2e-4
NUM_EPOCHS = 2

PER_DEVICE_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4

WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01

SEED = 3407

In [12]:
%%capture
# Install Unsloth and compatible dependencies for the Colab runtime
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.26" "trl<0.9.0" peft accelerator bitsandbytes

In [13]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1123 11:30:43.517000 352 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1123 11:30:43.557000 352 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.10: Fast Qwen3 patching. Transformers: 5.18.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]